# 💼 Notebook 4: Risk Scoring & Business Insights
## AI-Driven Credit Risk Evaluation
**Author:** Bandaru Yashwanth | B.Sc. Actuarial Science, Amity University Noida

---
### 🎯 Objective
Convert raw model probabilities into **actionable risk tiers** using actuarial classification principles. Generate business insights for the credit team dashboard.

> *This notebook bridges the gap between a trained ML model and real business decision-making — applying Actuarial Science principles to classify risk.*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
print('✅ Libraries loaded')

In [ ]:
# Load model, data, and original raw data
with open('../models/best_credit_risk_model.pkl', 'rb') as f:
    model = pickle.load(f)

X_test  = pd.read_csv('../data/X_test.csv')
y_test  = pd.read_csv('../data/y_test.csv').squeeze()
df_raw  = pd.read_csv('../data/credit_risk_raw.csv')

y_prob = model.predict_proba(X_test)[:, 1]
print(f'Model loaded | Test samples: {len(X_test):,}')
print(f'Probability range: {y_prob.min():.3f} — {y_prob.max():.3f}')

## 1️⃣ Actuarial Risk Scoring Framework
> In actuarial science, risk is classified into tiers for pricing and underwriting decisions. We apply the same principle to ML model outputs.

In [ ]:
def assign_risk_tier(prob):
    """
    Convert default probability to actuarial risk tier.
    Thresholds based on standard credit underwriting bands.
    """
    if prob < 0.10:   return 'LOW RISK',      '✅ Auto-Approve',              '#27AE60'
    elif prob < 0.25: return 'MODERATE RISK',  '📋 Standard Review',           '#F39C12'
    elif prob < 0.50: return 'HIGH RISK',       '⚠️  Senior Review Required',   '#E67E22'
    else:             return 'CRITICAL RISK',   '❌ Decline / Collateral',      '#E74C3C'

risk_df = pd.DataFrame({
    'default_probability': y_prob,
    'actual_default': y_test.values
})
risk_df[['risk_tier', 'recommended_action', 'color']] = pd.DataFrame(
    risk_df['default_probability'].apply(assign_risk_tier).tolist(),
    index=risk_df.index
)
risk_df['risk_score'] = (risk_df['default_probability'] * 1000).astype(int)

print('✅ Risk scoring complete')
print('\n📊 Risk Tier Distribution:')
print(risk_df['risk_tier'].value_counts())
print('\n📋 Sample Risk Profiles:')
risk_df[['default_probability','risk_score','risk_tier','recommended_action']].head(8)

## 2️⃣ Risk Tier Dashboard

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# --- Chart 1: Risk Tier Distribution ---
tier_order  = ['LOW RISK', 'MODERATE RISK', 'HIGH RISK', 'CRITICAL RISK']
tier_colors = ['#27AE60', '#F39C12', '#E67E22', '#E74C3C']
tier_counts = risk_df['risk_tier'].value_counts().reindex(tier_order, fill_value=0)

bars = axes[0].bar(range(len(tier_order)), tier_counts.values,
                   color=tier_colors, edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, tier_counts.values):
    pct = val / len(risk_df) * 100
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'{val:,}\n({pct:.1f}%)', ha='center', fontweight='bold', fontsize=9)
axes[0].set_xticks(range(len(tier_order)))
axes[0].set_xticklabels(['LOW', 'MODERATE', 'HIGH', 'CRITICAL'], fontsize=9)
axes[0].set_title('Customer Risk Tier Distribution', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Number of Customers')

# --- Chart 2: Actual Default Rate per Tier ---
tier_dr = risk_df.groupby('risk_tier')['actual_default'].mean().reindex(tier_order) * 100
bars2 = axes[1].bar(range(len(tier_order)), tier_dr.values,
                    color=tier_colors, edgecolor='black', linewidth=0.5)
for bar, val in zip(bars2, tier_dr.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.5,
                 f'{val:.1f}%', ha='center', fontweight='bold', fontsize=10)
axes[1].set_xticks(range(len(tier_order)))
axes[1].set_xticklabels(['LOW', 'MODERATE', 'HIGH', 'CRITICAL'], fontsize=9)
axes[1].set_title('Actual Default Rate per Risk Tier', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Actual Default Rate (%)')

# --- Chart 3: Probability Distribution ---
axes[2].hist(risk_df[risk_df['actual_default']==0]['default_probability'],
             bins=40, alpha=0.65, color='#27AE60', label='No Default', edgecolor='white')
axes[2].hist(risk_df[risk_df['actual_default']==1]['default_probability'],
             bins=40, alpha=0.65, color='#E74C3C', label='Default', edgecolor='white')
for thresh, ls, label in [(0.10,'--','Low/Mod'), (0.25,':','Mod/High'), (0.50,'-.','High/Critical')]:
    axes[2].axvline(thresh, color='navy', linestyle=ls, linewidth=1.5, label=f'Threshold {label}')
axes[2].set_title('Default Probability Distribution', fontweight='bold', fontsize=12)
axes[2].set_xlabel('Predicted Default Probability')
axes[2].set_ylabel('Count')
axes[2].legend(fontsize=8)

plt.suptitle('Credit Risk Dashboard — AI Model Output', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/risk_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

## 3️⃣ Business Impact Analysis

In [ ]:
print('=' * 65)
print('  💰 BUSINESS IMPACT ANALYSIS')
print('=' * 65)

# Financial assumptions
avg_loan     = df_raw['loan_amount'].mean()
loss_rate    = 0.60   # LGD: 60 cents lost per rupee defaulted (standard BFSI)

total_test   = len(risk_df)

# Without model: approve everyone → miss all defaults
actual_defaulters = risk_df['actual_default'].sum()
exposure_without  = actual_defaulters * avg_loan * loss_rate

# With model: flag HIGH + CRITICAL RISK → review/decline them
flagged_defaulters = risk_df[
    (risk_df['risk_tier'].isin(['HIGH RISK', 'CRITICAL RISK'])) &
    (risk_df['actual_default'] == 1)
].shape[0]

exposure_saved = flagged_defaulters * avg_loan * loss_rate
savings_pct    = (exposure_saved / exposure_without * 100) if exposure_without > 0 else 0

print(f'\n📊 Test Set Summary ({total_test:,} customers):')
print(f'   Total actual defaulters:      {actual_defaulters:,}')
print(f'   Average loan amount:          ₹{avg_loan:,.0f}')
print(f'   Loss Given Default (LGD):     {loss_rate*100:.0f}%')
print()
print(f'🚫 Without AI Model (approve all):')
print(f'   Financial exposure:           ₹{exposure_without:,.0f}')
print()
print(f'✅ With AI Model (flag High/Critical):')
print(f'   Defaulters flagged:           {flagged_defaulters:,} out of {actual_defaulters:,}')
print(f'   Exposure prevented:           ₹{exposure_saved:,.0f}')
print(f'   Risk reduction:               {savings_pct:.1f}%')
print()
print(f'💰 NET IMPACT: Model prevents ~{savings_pct:.0f}% of credit loss exposure')
print('=' * 65)

## 4️⃣ Predict for a New Customer (Demo)

In [ ]:
# Load feature columns used in training
feature_cols = X_test.columns.tolist()

# Create sample customer profile
sample_customer = pd.DataFrame(0, index=[0], columns=feature_cols)

# Fill in known values (numerical)
sample_customer['age']                  = 32
sample_customer['annual_income']        = 48000
sample_customer['loan_amount']          = 150000
sample_customer['loan_term_months']     = 48
sample_customer['credit_score']         = 610
sample_customer['employment_years']     = 2.5
sample_customer['num_credit_lines']     = 3
sample_customer['existing_debt']        = 35000
sample_customer['num_dependents']       = 2
sample_customer['debt_to_income_ratio'] = 35000 / 48000
sample_customer['loan_to_income_ratio'] = 150000 / 48000
sample_customer['monthly_emi_burden']   = 150000 / 48
sample_customer['emi_to_monthly_income']= (150000/48) / (48000/12)
sample_customer['financial_stress']     = 0.4*35000/48000 + 0.3*150000/48000 + 0.3*(150000/48)/(48000/12)
sample_customer['credit_tier']          = 2   # Poor (610 score)
sample_customer['income_tier']          = 2   # Lower-Mid
sample_customer['age_group']            = 2   # Early Career

# Set employment type (Salaried = drop_first handled → Salaried is baseline, set others to 0)
if 'employment_type_Self-Employed' in feature_cols:
    sample_customer['employment_type_Self-Employed'] = 0

prob = model.predict_proba(sample_customer)[0][1]
tier, action, color = assign_risk_tier(prob)
score = int(prob * 1000)

print('╔══════════════════════════════════════════════╗')
print('║         CREDIT RISK ASSESSMENT REPORT        ║')
print('╠══════════════════════════════════════════════╣')
print(f'║  Customer Profile:                           ║')
print(f'║    Age: 32 | Income: ₹48,000/yr              ║')
print(f'║    Loan: ₹1,50,000 for 48 months             ║')
print(f'║    Credit Score: 610 | Existing Debt: ₹35K   ║')
print(f'║    Employment: 2.5 yrs | Salaried            ║')
print('╠══════════════════════════════════════════════╣')
print(f'║  🎯 Default Probability: {prob*100:5.1f}%              ║')
print(f'║  📊 Risk Score:          {score:4d} / 1000           ║')
print(f'║  🏷️  Risk Tier:           {tier:<22}║')
print(f'║  📋 Recommendation:      {action:<22}║')
print('╚══════════════════════════════════════════════╝')

In [ ]:
print('\n' + '=' * 55)
print('  🏆 PROJECT COMPLETE — SUMMARY')
print('=' * 55)
print()
print('📊 Dataset:      10,000 customer financial records')
print('⚙️  Features:     Original + 8 actuarial engineered features')
print('🤖 Best Model:   Gradient Boosting Classifier')
print('📈 ROC-AUC:      0.84+ (strong performance)')
print('💡 Innovation:   Actuarial risk tiers applied to ML output')
print('💰 Impact:       Significant credit loss exposure reduction')
print()
print('📁 Files Generated:')
print('  data/credit_risk_raw.csv        → Raw dataset')
print('  data/X_train/X_test.csv         → Processed splits')
print('  data/risk_dashboard.png         → Power BI ready chart')
print('  data/roc_curves.png             → Model comparison')
print('  data/feature_importance.png     → Top features')
print('  models/best_credit_risk_model.pkl → Deployable model')
print()
print('Author: Bandaru Yashwanth')
print('B.Sc. Actuarial Science | Amity University Noida | Merit Scholar')